In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses

In [ ]:
MAX_VOCAB = 1000
CONTEXT_WIN = 20
EMBED_WIN = 64
HEADS = 2
FEED_FOWARD = 64
TRANSFORMER_BLOCKS = 2
BATCH_SIZE = 16
EPOCHS = 50

train_text = [
"the sun rises in the east",
"the sun sets in the west",
"the moon shines at night",
"the stars shine at night",
"the sky is blue in the day",
"the sky is dark at night",
"alice likes apples",
"alice likes bananas",
"alice likes oranges",
"bob likes apples",
"bob likes bananas",
"bob likes oranges",
"carol likes apples",
"carol likes bananas",
"carol likes oranges",
"alice eats apples every day",
"bob eats bananas every day",
"carol eats oranges every day",
"red is a color",
"blue is a color",
"green is a color",
"yellow is a color",
"cats chase mice",
"dogs chase cats",
"mice eat cheese",
"birds eat seeds",
"one two three four five",
"two three four five six",
"three four five six seven",
"four five six seven eight",
"if it rains the ground is wet",
"if the sun shines the sky is bright",
"if it is night the stars appear",
"if it is day the sun appears",
"hello world",
"hello alice",
"hello bob",
"hello carol",
"goodbye world",
"goodbye alice",
"goodbye bob",
"goodbye carol"
]

In [ ]:
tokenizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=CONTEXT_WIN
)

tokenizer.adapt(train_text)
vocab = tokenizer.get_vocabulary()

#next token prediction
def prep_data(train_text):
  seq = tokenizer(train_text)
  x_input = seq[:, :-1]
  y_target = seq[:, 1:]

x_train, y train = prep_data(train_text)

In [ ]:
def perplexity(true,pred):
  return tf.exp(tf.reduce_mean(losses.sparse_categorical_crossentropy(true,pred)))

class TokenPositionEmbedding(layers.Layer):
  def __init__(self, context_win, max_vocab, embed_dim, **kwargs) -> None:
    super().__init__()
    self.token_embed = layers.Embedding(input_dim = max_vocab, output_dim=embed_dim)
    self.position_embed = layers.Embedding(input_dim = context_win, output_dim=embed_dim)

  def call(self, x):
    context_win = tf.shape(x)[-1]
    positions = self.position_embed(tf.range(start=0, limit=context_win, delta=1))
    return self.token_embed(x) + positions


Each head computes:

attention(Q,K,V) = softmax(QK/sqrt(d_k))*V

basically word context

transformer block:

Input

   │

   ├── Self Attention

   ├── Add (Residual)

   ├── LayerNorm

   │

   ├── Feed Forward

   ├── Add (Residual)

   ├── LayerNorm

   │

Output



In [ ]:
class TransformerBlock(layers.Layer):
  def __init__(self, embed_dim, heads, feed_forward, **kwargs) -> None:
    super().__init__()
    self.attention = layers.MultiHeadAttention(num_heads=heads, key_dim=embed_dim)
    self.feed_foward_net = models.Sequential([layers.Dense(feed_forward, activation="relu"), layers.Dense(embed_dim)])
    self.norm1 = layers.LayerNormalization(epsilon=1e-4)
    self.norm2 = layers.LayerNormalization(epsilon=1e-4)
    self.drop1 = layers.Dropout(0.1)
    self.drop2 = layers.Dropout(0.1)

  def call(self, inputs, training = False):
    #Self Attention
    attention_output = self.attention(inputs, inputs, use_causal_mask=True)
    attention_output = self.drop1(attention_output, training=training)
    output = self.norm1(inputs + attention_output)

    #Feed Forward Network
    feed_forward_output = self.feed_foward_net(output)
    feed_foward_output = self.drop2(feed_forward_output, training=training)
    return self.norm2(output + feed_forward_output)